In [ ]:
##################### Main code to train model initially and store it ###############################
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import cv2
import os
from sklearn.model_selection import train_test_split
from datetime import datetime

CAPTCHA_LENGTH = 6
CHARACTERS = '0123456789'

def encode_label(text):
    return [CHARACTERS.index(c) for c in text]

def decode_label(label):
    return ''.join(CHARACTERS[c] for c in label)

def load_data(folder):
    X, y = [], []
    for filename in os.listdir(folder):
        if filename.endswith('.png'):
            label = filename.split('.')[0]
            img = cv2.imread(os.path.join(folder, filename), cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (200, 50)) / 255.0
            X.append(img.reshape(50, 200, 1))
            y.append(encode_label(label))
    return np.array(X), np.array(y)

# Load data
X, y = load_data('captcha_dataset/')
y = tf.keras.utils.to_categorical(y, num_classes=len(CHARACTERS))
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1)

# Define model
def build_model():
    inputs = layers.Input(shape=(50, 200, 1))
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(1024, activation='relu')(x)

    outputs = [layers.Dense(len(CHARACTERS), activation='softmax')(x) for _ in range(CAPTCHA_LENGTH)]
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer='adam',
        loss=['categorical_crossentropy'] * CAPTCHA_LENGTH,
        metrics=['accuracy'] * CAPTCHA_LENGTH
    )
    return model


model = build_model()
model.fit(X_train, [y_train[:, i] for i in range(CAPTCHA_LENGTH)],
          validation_data=(X_val, [y_val[:, i] for i in range(CAPTCHA_LENGTH)]),
          epochs=20, batch_size=32)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model.save(f'captcha_model_updated_{timestamp}.h5')


Epoch 1/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 63s 682ms/step - dense_1_accuracy: 0.2044 - dense_1_loss: 2.2273 - dense_2_accuracy: 0.1599 - dense_2_loss: 2.3352 - dense_3_accuracy: 0.1574 - dense_3_loss: 2.3440 - dense_4_accuracy: 0.1796 - dense_4_loss: 2.3661 - dense_5_accuracy: 0.1941 - dense_5_loss: 2.3280 - dense_6_accuracy: 0.1499 - dense_6_loss: 2.3689 - loss: 13.9704 - val_dense_1_accuracy: 0.9000 - val_dense_1_loss: 0.3175 - val_dense_2_accuracy: 0.8167 - val_dense_2_loss: 0.6773 - val_dense_3_accuracy: 0.6967 - val_dense_3_loss: 1.0152 - val_dense_4_accuracy: 0.7133 - val_dense_4_loss: 0.9948 - val_dense_5_accuracy: 0.6967 - val_dense_5_loss: 0.9095 - val_dense_6_accuracy: 0.5267 - val_dense_6_loss: 1.3027 - val_loss: 5.2216
Epoch 2/20
85/85 ━━━━━━━━━━━━━━━━━━━━ 58s 683ms/step - dense_1_accuracy: 0.9648 - dense_1_loss: 0.1486 - dense_2_accuracy: 0.8906 - dense_2_loss: 0.4002 - dense_3_accuracy: 0.7977 - dense_3_loss: 0.6544 - dense_4_accuracy: 0.8196 - dense_4_loss: 0.5946 - dense_5_a

In [ ]:
###### Code to load existing saved trained model ##########################################

# from tensorflow.keras.models import load_model
# model = load_model('captcha_model_updated_20250530_181802.h5')

In [ ]:
############### Code to download capthas and store them after labeling with output #########
import requests
import os
from tensorflow.keras import layers, models
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from datetime import datetime

CAPTCHA_LENGTH = 6
CHARACTERS = '0123456789'

# Create directory for downloads
os.makedirs('captcha_downloads', exist_ok=True)

# Example for a hypothetical captcha generator URL
base_url = "https://services.gst.gov.in/services/captcha?rnd=0.920321416851637"  # Replace with actual base URL

for i in range(1, 11):  # Download 50 images
    try:
        # Modify URL parameters if needed (some sites use session IDs)
        url = f"{base_url}id={i}"
        response = requests.get(url, stream=True, timeout=10)
        
        if response.status_code == 200:
            with open(f'captcha_downloads/captcha_{i}.png', 'wb') as f:
                f.write(response.content)
            print(f"Downloaded captcha_{i}.png")
        else:
            print(f"Failed to download image {i}")
    except Exception as e:
        print(f"Error downloading image {i}: {str(e)}")



# Block to load trained model 
# from tensorflow.keras.models import load_model
# model = load_model('captcha_model.h5') 

def predict_captcha(model, image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (200, 50)) / 255.0
    img = img.reshape(1, 50, 200, 1)
    preds = model.predict(img)
    # print(preds)
    pred_text = ''.join(CHARACTERS[np.argmax(p)] for p in preds)
    return pred_text

# print(predict_captcha(model, r'C:\Users\PC\Pictures\Saved Pictures\captcha.png'))
# print(predict_captcha(model, r'D:\Nexensus_Projects\red_line_removed_captcha.png'))
# for i in range(1, 31):
#     print(r'C:\Users\PC\Pictures\Saved Pictures\captcha ({}).png'.format(i))
#     print(i,"  ", predict_captcha(model, r'C:\Users\PC\Pictures\Saved Pictures\captcha_{}.png'.format(i)))

import os
# Adjust your folder path accordingly
folder = r'captcha_downloads'

for i in range(1, 11):
    # Original filename
    original_filename = os.path.join(folder, f'captcha_{i}.png')

    # Get predicted label
    predicted_label = predict_captcha(model, original_filename)  # e.g., "232424"
    
    # New filename based on prediction
    new_filename = os.path.join(folder, f'{predicted_label}.png')

    # Rename the file
    if os.path.exists(original_filename):
        os.rename(original_filename, new_filename)
        print(f'Renamed: {original_filename} → {new_filename}')
    else:
        print(f'❌ File not found: {original_filename}')


Downloaded captcha_1.png
Downloaded captcha_2.png
Downloaded captcha_3.png
Downloaded captcha_4.png
Downloaded captcha_5.png
Downloaded captcha_6.png
Downloaded captcha_7.png
Downloaded captcha_8.png
Downloaded captcha_9.png
Downloaded captcha_10.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Renamed: captcha_downloads\captcha_1.png → captcha_downloads\940895.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
Renamed: captcha_downloads\captcha_2.png → captcha_downloads\149410.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
Renamed: captcha_downloads\captcha_3.png → captcha_downloads\620147.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
Renamed: captcha_downloads\captcha_4.png → captcha_downloads\523272.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Renamed: captcha_downloads\captcha_5.png → captcha_downloads\917444.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
Renamed: captcha_downloads\captcha_6.png → captcha_downloads\550615.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
Renamed: captcha_downloads\captcha_7.png → captcha_

In [ ]:
##### Code to train model already trained with additional/new dataset Downloaded and labeled #############

from tensorflow.keras.optimizers import Adam

# Use a smaller learning rate (e.g., 0.0001 instead of the default 0.001)
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=['categorical_crossentropy'] * CAPTCHA_LENGTH,
    metrics=['accuracy'] * CAPTCHA_LENGTH
)

import tensorflow as tf
from tensorflow.keras.models import load_model
import numpy as np
import cv2
import os
from sklearn.model_selection import train_test_split
from datetime import datetime

# Constants
CAPTCHA_LENGTH = 6
CHARACTERS = '0123456789'

# Label encoding/decoding
def encode_label(text):
    return [CHARACTERS.index(c) for c in text]

def load_data(folder):
    X, y = [], []
    for filename in os.listdir(folder):
        if filename.endswith('.png'):
            label = filename.split('.')[0]
            if len(label) != CAPTCHA_LENGTH:
                continue
            img = cv2.imread(os.path.join(folder, filename), cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (200, 50)) / 255.0
            X.append(img.reshape(50, 200, 1))
            y.append(encode_label(label))
    return np.array(X), np.array(y)

# Load new data
X_new, y_new = load_data('captcha_downloads/')  # Replace with your new dataset folder
y_new = tf.keras.utils.to_categorical(y_new, num_classes=len(CHARACTERS))
X_train, X_val, y_train, y_val = train_test_split(X_new, y_new, test_size=0.1)

# Load the pre-trained model
# model = load_model('captcha_model.h5')

# Continue training the model
model.fit(
    X_train, [y_train[:, i] for i in range(CAPTCHA_LENGTH)],
    validation_data=(X_val, [y_val[:, i] for i in range(CAPTCHA_LENGTH)]),
    epochs=10,  # You can adjust the number of additional epochs
    batch_size=32
)

# Save the updated model
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model.save(f'captcha_model_updated_{timestamp}.h5')


Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - dense_1_accuracy: 1.0000 - dense_1_loss: 1.3368e-04 - dense_2_accuracy: 1.0000 - dense_2_loss: 0.0088 - dense_3_accuracy: 1.0000 - dense_3_loss: 0.0808 - dense_4_accuracy: 0.8889 - dense_4_loss: 0.7226 - dense_5_accuracy: 1.0000 - dense_5_loss: 0.0125 - dense_6_accuracy: 1.0000 - dense_6_loss: 0.0039 - loss: 0.8286 - val_dense_1_accuracy: 1.0000 - val_dense_1_loss: 0.0065 - val_dense_2_accuracy: 1.0000 - val_dense_2_loss: 2.8610e-06 - val_dense_3_accuracy: 1.0000 - val_dense_3_loss: 0.0000e+00 - val_dense_4_accuracy: 1.0000 - val_dense_4_loss: 1.1921e-07 - val_dense_5_accuracy: 1.0000 - val_dense_5_loss: 0.0000e+00 - val_dense_6_accuracy: 1.0000 - val_dense_6_loss: 0.0000e+00 - val_loss: 0.0065
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 709ms/step - dense_1_accuracy: 1.0000 - dense_1_loss: 9.0658e-05 - dense_2_accuracy: 1.0000 - dense_2_loss: 0.0054 - dense_3_accuracy: 1.0000 - dense_3_loss: 0.0053 - dense_4_accuracy: 0.8889 - dense_4_loss: 

Screenshot saved as captcha.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
wrong captcha
Screenshot saved as captcha.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
Correct Captcha


In [ ]:
############### Code to predict captcha validation and storing captcha in dataset for further training ##########

import os
import time
import shutil
import csv
import json
import os.path
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager  # Optional, for automatic driver management
from tensorflow.keras.models import load_model
import requests
from tensorflow.keras import layers, models
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from datetime import datetime

CAPTCHA_LENGTH = 6
CHARACTERS = '0123456789'
###### Code to load existing saved trained model ##########################################
model = load_model(r'D:\Nexensus_Projects\GST_TaxPayer\captcha_model_updated_20250602_131721.h5')

## Function to predict captcha text ####################################################################
def predict_captcha(model, image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (200, 50)) / 255.0
    img = img.reshape(1, 50, 200, 1)
    preds = model.predict(img)
    # print(preds)
    pred_text = ''.join(CHARACTERS[np.argmax(p)] for p in preds)
    return pred_text

# Create a Service object
service = Service(ChromeDriverManager().install())  # or specify your path directly here

# Pass the Service object to the WebDriver
driver = webdriver.Chrome(service=service)

url = "https://services.gst.gov.in/services/searchtpbypan"
pan = "AAACT2727Q"
driver.get(url)
# time.sleep(0.5)
driver.maximize_window()
time.sleep(3)

driver.find_element(By.ID, 'for_gstin').send_keys(pan)
time.sleep(3)

while True:
    captcha_element = driver.find_element(By.ID, 'imgCaptcha')
    os.makedirs('resolved_captchaset', exist_ok=True)
    os.makedirs('unresolved_captchaset', exist_ok=True)

    # Temporary file to hold the screenshot before naming
    temp_path = r'D:\Nexensus_Projects\GST_TaxPayer\captcha_screenshots\captcha.png'
    captcha_element.screenshot(temp_path)
    print("Screenshot saved temporarily as captcha.png")

    time.sleep(1)
    captcha_value = predict_captcha(model, temp_path)
    time.sleep(1)

    driver.find_element(By.ID, 'fo-captcha').send_keys(captcha_value)
    time.sleep(1)
    driver.find_element(By.ID, "lotsearch").click()
    time.sleep(3)

    # Target filename based on predicted text
    filename = f"{captcha_value}.png"

    if not any(i.text == "Enter valid letters shown in the image below" for i in driver.find_elements(By.CLASS_NAME, 'err')):
        print("Correct Captcha")
        shutil.move(temp_path, os.path.join('resolved_captchaset', filename))
        break
    else:
        print("Wrong Captcha")
        shutil.move(temp_path, os.path.join('unresolved_captchaset', filename))

driver.quit()


Screenshot saved temporarily as captcha.png


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
Wrong Captcha
Screenshot saved temporarily as captcha.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
Correct Captcha


In [ ]:
################## Block to fetch gstin using pan as input #################################################
import requests
import time
import random

def fetch_gstin_list(pan_number):
    session = requests.Session()
    rnd = random.random()
    captcha_url = f'https://services.gst.gov.in/services/captcha?rnd={rnd}'
    captcha_response = session.get(captcha_url)
    if captcha_response.status_code != 200:
        print("Failed to fetch captcha")
        return

    with open("captcha_image.png", "wb") as f:
        f.write(captcha_response.content)
    print("📷 Captcha saved as 'captcha_image.png'. Please open and solve it.")
    captcha_text = input("🔑 Enter captcha: ")
    api_url = 'https://services.gst.gov.in/services/api/get/gstndtls'
    headers = {
        'Content-Type': 'application/json',
        'User-Agent': 'Mozilla/5.0',
        'Referer': 'https://services.gst.gov.in/services/searchtpbypan',
        'Origin': 'https://services.gst.gov.in'
    }
    payload = {
        'panNO': pan_number,
        'captcha': captcha_text
    }
    response = session.post(api_url, json=payload, headers=headers)

    print(f"Status Code: {response.status_code}")
    print("Response Text:")
    print(response.text)

    if response.status_code == 200:
        data = response.json()
        print(data.keys())
        print(data['gstinResList'])
        # if data.get('status') == 'Success':
        #     print("GSTIN List:")
        #     for gstin in data.get('data', []):
        #         print(gstin)
        # else:
        #     print("No GSTINs found or error:", data.get('message'))
    else:
        print("Request failed.")

for pan in ["AAACT2727Q", "AACCS7101B", "AALCA7867M"]:
    fetch_gstin_list(pan)  # Replace with actual PAN

📷 Captcha saved as 'captcha_image.png'. Please open and solve it.
Status Code: 200
Response Text:
{"panNum":null,"action":null,"gstinResList":[{"gstin":"29AAACT2727Q1ZS","authStatus":"Active","stateCd":"29"},{"gstin":"07AAACT2727Q1ZY","authStatus":"Active","stateCd":"07"},{"gstin":"34AAACT2727Q1Z1","authStatus":"Inactive","stateCd":"34"},{"gstin":"26AAACT2727Q1ZY","authStatus":"Active","stateCd":"26"},{"gstin":"22AAACT2727Q1Z6","authStatus":"Active","stateCd":"22"},{"gstin":"14AAACT2727Q1Z3","authStatus":"Inactive","stateCd":"14"},{"gstin":"27AAACT2727Q2ZV","authStatus":"Active","stateCd":"27"},{"gstin":"24AAACT2727Q1Z2","authStatus":"Active","stateCd":"24"},{"gstin":"20AAACT2727Q1ZA","authStatus":"Active","stateCd":"20"},{"gstin":"27AAACT2727Q1ZW","authStatus":"Active","stateCd":"27"},{"gstin":"37AAACT2727Q1ZV","authStatus":"Active","stateCd":"37"},{"gstin":"03AAACT2727Q1Z6","authStatus":"Active","stateCd":"03"},{"gstin":"30AAACT2727Q1Z9","authStatus":"Active","stateCd":"30"},{"gstin"

KeyError: 'gstinResList'

In [ ]:
############### Block to get further details by gstin ########################################################
import requests
import time
import random

def fetch_gstin_list(gstin):
    session = requests.Session()
    rnd = random.random()
    captcha_url = f'https://services.gst.gov.in/services/captcha?rnd={rnd}'
    captcha_response = session.get(captcha_url)
    if captcha_response.status_code != 200:
        print("Failed to fetch captcha")
        return

    with open("captcha_image.png", "wb") as f:
        f.write(captcha_response.content)
    print("📷 Captcha saved as 'captcha_image.png'. Please open and solve it.")
    captcha_text = input("🔑 Enter captcha: ")
    api_url = 'https://services.gst.gov.in/services/api/search/taxpayerDetails'
    headers = {
        'Content-Type': 'application/json',
        'User-Agent': 'Mozilla/5.0',
        'Referer': 'https://services.gst.gov.in/services/searchtp',
        'Origin': 'https://services.gst.gov.in'
    }
    payload = {
        'gstin': gstin,
        'captcha': captcha_text
    }
    response = session.post(api_url, json=payload, headers=headers)

    print(f"Status Code: {response.status_code}")
    print("Response Text:")
    print(response.text)

    if response.status_code == 200:
        data = response.json()
        print(data.keys())
        print(data)
        # print(data['gstinResList'])
    else:
        print("Request failed.")

for gstin in ["27AAMCM7090B1ZC", "37AAMCM7090B1ZB", "10AAECS8975P1ZO", "27AAECS8975P1Z9", "18AAECS8975P1Z8", "21AAECS8975P1ZL", "19AAECS8975P2Z5", "19AAECS8975P1Z6"]:
    fetch_gstin_list(gstin)  # Replace with actual PAN

📷 Captcha saved as 'captcha_image.png'. Please open and solve it.
Status Code: 200
Response Text:
{"ntcrbs":"SPO","adhrVFlag":"Yes","lgnm":"MODRIG TECHSOLUTIONS PRIVATE LIMITED","stj":"State - Maharashtra,Zone - RAIGAD,Division - RAIGAD_SOUTH,Charge - SANPADA_506 (Jurisdictional Office)","dty":"Regular","cxdt":"","gstin":"27AAMCM7090B1ZC","nba":["Recipient of Goods or Services"],"ekycVFlag":"Not Applicable","cmpRt":"NA","rgdt":"14/08/2019","ctb":"Private Limited Company","pradr":{"adr":"Plot No. W-65, MIDC T.T.C. INDL. Area,, RABALE Village, Ghansoli, Than, Plot No. W-65, MIDC T.T.C. INDL. Area,, Thane, Maharashtra, 400701"},"sts":"Active","tradeNam":"NA","isFieldVisitConducted":"No","adhrVdt":"05/06/2025","ctj":"State - CBIC,Zone - MUMBAI,Commissionerate - BELAPUR,Division - DIVISION IV,Range - RANGE-III","einvoiceStatus":"Yes"}
dict_keys(['ntcrbs', 'adhrVFlag', 'lgnm', 'stj', 'dty', 'cxdt', 'gstin', 'nba', 'ekycVFlag', 'cmpRt', 'rgdt', 'ctb', 'pradr', 'sts', 'tradeNam', 'isFieldVisit

In [1]:
############### Block to get further details by gstin ########################################################
import requests
import time
import random

def fetch_gstin_list(gstin):
    session = requests.Session()
    # rnd = random.random()
    # captcha_url = f'https://services.gst.gov.in/services/captcha?rnd={rnd}'
    # captcha_response = session.get(captcha_url)
    # if captcha_response.status_code != 200:
    #     print("Failed to fetch captcha")
    #     return

    # with open("captcha_image.png", "wb") as f:
    #     f.write(captcha_response.content)
    # print("📷 Captcha saved as 'captcha_image.png'. Please open and solve it.")
    # captcha_text = input("🔑 Enter captcha: ")
    api_url = 'https://services.gst.gov.in/services/api/search/goodservice?gstin={}'.format(gstin)
    headers = {
        'Content-Type': 'application/json',
        'User-Agent': 'Mozilla/5.0',
        'Referer': 'https://services.gst.gov.in/services/searchtp',
        'Origin': 'https://services.gst.gov.in'
    }
    payload = {
        'gstin': gstin,
    }
    response = session.get(api_url, json=payload, headers=headers)

    print(f"Status Code: {response.status_code}")
    print("Response Text:")
    print(response.text)

    if response.status_code == 200:
        data = response.json()
        print(data.keys())
        print(data)
        # print(data['gstinResList'])
    else:
        print("Request failed.")

for gstin in ["27AAMCM7090B1ZC", "37AAMCM7090B1ZB", "10AAECS8975P1ZO", "27AAECS8975P1Z9", "18AAECS8975P1Z8", "21AAECS8975P1ZL", "19AAECS8975P2Z5", "19AAECS8975P1Z6"]:
    fetch_gstin_list(gstin)  # Replace with actual PAN

Status Code: 200
Response Text:
{"bzsdtls":[{"saccd":"998399","sdes":"Other professional, technical and business services nowhere else classified"}]}
dict_keys(['bzsdtls'])
{'bzsdtls': [{'saccd': '998399', 'sdes': 'Other professional, technical and business services nowhere else classified'}]}
Status Code: 200
Response Text:
{"bzgddtls":[{"gdes":"TOOLS FOR WORKING IN THE HAND, PNEUMATIC, HYDRAULIC OR WITH SELF-CONTAINED ELECTRIC OR NON-ELECTRIC MOTOR","hsncd":"8467"},{"gdes":"INSTRUMENTS AND APPARATUS FOR MEASURING OR CHECKING THE FLOW, LEVEL, PRESSURE OR OTHER VARIABLES OF LIQUIDS OR GASES (FOR EXAMPLE, FLOW METERS, LEVEL GAUGES, MANOMETERS, HEAT METERS), EXCLUDING INSTRUMENTS AND APPARATUS OF HEADING 9014, 9015, 9028 OR 9032","hsncd":"9026"},{"gdes":"TUBE OR PIPE FITTINGS (FOR EXAMPLE, COUPLINGS, ELBOWS, SLEEVES), OF IRON OR STEEL","hsncd":"7307"}],"bzsdtls":[{"saccd":"998399","sdes":"Other professional, technical and business services nowhere else classified"},{"saccd":"998513","sde

In [ ]:
############## Code for fetching filling frequency data for current and previous year ##################################
for i in requests.get("https://services.gst.gov.in/services/api/dropdownfinyear?gstin=07AAGCB2181A1Z7").json()['data']:
    print(i['value'])
    payload = {"gstin":"07AAGCB2181A1Z7","fy":str(i['value'])}
    print(payload)
    print(requests.post("https://services.gst.gov.in/services/api/search/taxpayerReturnDetails", json=payload).json())

2017
{'gstin': '07AAGCB2181A1Z7', 'fy': '2017'}
{'filingStatus': [[{'fy': '2017-2018', 'taxp': 'March', 'mof': 'ONLINE', 'dof': '29/03/2019', 'rtntype': 'GSTR1', 'arn': 'NA', 'status': 'Filed'}, {'fy': '2017-2018', 'taxp': 'February', 'mof': 'ONLINE', 'dof': '27/10/2018', 'rtntype': 'GSTR1', 'arn': 'NA', 'status': 'Filed'}, {'fy': '2017-2018', 'taxp': 'January', 'mof': 'ONLINE', 'dof': '26/10/2018', 'rtntype': 'GSTR1', 'arn': 'NA', 'status': 'Filed'}, {'fy': '2017-2018', 'taxp': 'December', 'mof': 'ONLINE', 'dof': '26/10/2018', 'rtntype': 'GSTR1', 'arn': 'NA', 'status': 'Filed'}, {'fy': '2017-2018', 'taxp': 'November', 'mof': 'ONLINE', 'dof': '22/10/2018', 'rtntype': 'GSTR1', 'arn': 'NA', 'status': 'Filed'}, {'fy': '2017-2018', 'taxp': 'October', 'mof': 'ONLINE', 'dof': '22/10/2018', 'rtntype': 'GSTR1', 'arn': 'NA', 'status': 'Filed'}, {'fy': '2017-2018', 'taxp': 'September', 'mof': 'ONLINE', 'dof': '10/10/2018', 'rtntype': 'GSTR1', 'arn': 'NA', 'status': 'Filed'}, {'fy': '2017-2018', 

In [3]:
############## Code for fetching filling table data ##################################
import requests
for gstin in ["27AAMCM7090B1ZC", "37AAMCM7090B1ZB", "10AAECS8975P1ZO", "27AAECS8975P1Z9", "18AAECS8975P1Z8", "21AAECS8975P1ZL", "19AAECS8975P2Z5", "19AAECS8975P1Z6"]:
    print(gstin)
    for Year in ['2024', '2025']:
        response = requests.get("https://services.gst.gov.in/services/api/search/taxpayerProfileDetails?fy={}&gstin={}".format(Year, gstin)).json()
        if "error" in response:
            print(response['error']['message']) 
        else:
            print(response)
        print("-------------")

27AAMCM7090B1ZC
{'status': 1, 'data': {'response': [{'quarter': 'Q1', 'preference': 'M'}, {'quarter': 'Q2', 'preference': 'M'}, {'quarter': 'Q3', 'preference': 'M'}, {'quarter': 'Q4', 'preference': 'M'}]}}
-------------
{'status': 1, 'data': {'response': [{'quarter': 'Q1', 'preference': 'M'}, {'quarter': 'Q2', 'preference': 'M'}, {'quarter': 'Q3', 'preference': 'M'}, {'quarter': 'Q4', 'preference': 'M'}]}}
-------------
37AAMCM7090B1ZB
{'status': 1, 'data': {'response': [{'quarter': 'Q1', 'preference': 'M'}, {'quarter': 'Q2', 'preference': 'M'}, {'quarter': 'Q3', 'preference': 'M'}, {'quarter': 'Q4', 'preference': 'M'}]}}
-------------
{'status': 1, 'data': {'response': [{'quarter': 'Q1', 'preference': 'M'}, {'quarter': 'Q2', 'preference': 'M'}, {'quarter': 'Q3', 'preference': 'M'}, {'quarter': 'Q4', 'preference': 'M'}]}}
-------------
10AAECS8975P1ZO
{'status': 1, 'data': {'response': [{'quarter': 'Q1', 'preference': 'M'}, {'quarter': 'Q2', 'preference': 'M'}, {'quarter': 'Q3', 'prefe

In [1]:
################# Code to navigate through csv file containing all the gsting to get active ones of a particular PAN ##################
import json
import pandas as pd

df = pd.read_csv(r"D:\Nexensus_Projects\GST_TaxPayer\panDetails_ScrapedData.csv")
for i in range(len(df)):
    print("total:", len(json.loads(df['GSTIN'][i])['gstinResList']))
    print(json.loads(df['GSTIN'][i])['gstinResList'])
    for j in json.loads(df['GSTIN'][i])['gstinResList']:
        if j['authStatus'].lower() == "active":
            print(j['gstin'])
            break


    print("-----------------------------")

total: 6
[{'gstin': '10AAECS8975P1ZO', 'authStatus': 'Active', 'stateCd': '10'}, {'gstin': '27AAECS8975P1Z9', 'authStatus': 'Active', 'stateCd': '27'}, {'gstin': '18AAECS8975P1Z8', 'authStatus': 'Active', 'stateCd': '18'}, {'gstin': '21AAECS8975P1ZL', 'authStatus': 'Active', 'stateCd': '21'}, {'gstin': '19AAECS8975P2Z5', 'authStatus': 'Active', 'stateCd': '19'}, {'gstin': '19AAECS8975P1Z6', 'authStatus': 'Active', 'stateCd': '19'}]
10AAECS8975P1ZO
-----------------------------
total: 6
[{'gstin': '09AABCA7448J1ZE', 'authStatus': 'Active', 'stateCd': '09'}, {'gstin': '37AABCA7448J2ZE', 'authStatus': 'Inactive', 'stateCd': '37'}, {'gstin': '36AABCA7448J2ZG', 'authStatus': 'Active', 'stateCd': '36'}, {'gstin': '36AABCA7448J1ZH', 'authStatus': 'Active', 'stateCd': '36'}, {'gstin': '37AABCA7448J1ZF', 'authStatus': 'Inactive', 'stateCd': '37'}, {'gstin': '27AABCA7448J1ZG', 'authStatus': 'Active', 'stateCd': '27'}]
09AABCA7448J1ZE
-----------------------------
total: 6
[{'gstin': '10AAECS8975

In [ ]:
########################## Selenium Stealth demo code to minimize detection while scraping ############
from selenium import webdriver
from selenium_stealth import stealth
import time

options = webdriver.ChromeOptions()
options.add_argument("start-maximized")

# options.add_argument("--headless")

options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
driver = webdriver.Chrome(options=options)

stealth(driver,
    user_agent= 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/83.0.4103.53 Safari/537.36',
    languages= ["en-US", "en"],
    vendor= "Google Inc.",
    platform = "Win32",
    webgl_vendor = "Intel Inc.",
    renderer = "Intel Iris OpenGL Engine",
    fix_hairline = False,
    run_on_insecure_origins = False,
        )

# url = "https://bot.sannysoft.com/"
url = "https://services.gst.gov.in/services/searchtp"
driver.get(url)
time.sleep(5)
driver.quit()